# Parte 2 — Preparacion de datos y modelos predictivos

## 1. Carga del dataset tratado

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)
from imblearn.over_sampling import SMOTE
from sklearn.inspection import permutation_importance

sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('TelecomX_limpio.csv')
print(df.shape)
df.head()

## 2. Eliminacion de columnas sin valor predictivo

In [ ]:
df.drop(columns=['id_cliente', 'cargo_total', 'cargo_diario'], inplace=True)
print('Columnas restantes:', df.columns.tolist())
print('Shape:', df.shape)

## 3. Codificacion de variables categoricas (One-Hot Encoding)

In [ ]:
cols_categoricas = df.select_dtypes(include='object').columns.tolist()
print('Columnas categoricas:', cols_categoricas)

In [ ]:
df_encoded = pd.get_dummies(df, columns=cols_categoricas, drop_first=False, dtype=int)
print('Shape post-encoding:', df_encoded.shape)
df_encoded.head()

## 4. Analisis de desbalance de clases

In [ ]:
conteo = df_encoded['cancelacion'].value_counts()
proporcion = df_encoded['cancelacion'].value_counts(normalize=True) * 100

print('Distribucion de clases:')
print(pd.DataFrame({'Cantidad': conteo, 'Porcentaje': proporcion.round(2)}))

fig, ax = plt.subplots(figsize=(5, 4))
colores = ['#2ecc71', '#e74c3c']
ax.bar(['Activo (0)', 'Cancelo (1)'], conteo.values, color=colores, edgecolor='black', width=0.5)
for i, v in enumerate(conteo.values):
    ax.text(i, v + 30, f'{v}\n({proporcion.values[i]:.1f}%)', ha='center', fontweight='bold')
ax.set_title('Distribucion de la variable objetivo', fontweight='bold')
ax.set_ylabel('Cantidad de clientes')
plt.tight_layout()
plt.show()

## 5. Balanceo de clases con SMOTE

In [ ]:
X = df_encoded.drop(columns=['cancelacion'])
y = df_encoded['cancelacion']

X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_raw, y_train_raw)

print('Antes de SMOTE:', y_train_raw.value_counts().to_dict())
print('Despues de SMOTE:', pd.Series(y_train_bal).value_counts().to_dict())

## 6. Normalizacion de datos

In [ ]:
# Se aplica StandardScaler para modelos sensibles a la escala (Regresion Logistica).
# Random Forest no requiere normalizacion, pero se usa el mismo split para comparacion justa.

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled  = scaler.transform(X_test_raw)

print('Datos de entrenamiento (balanceados):', X_train_scaled.shape)
print('Datos de prueba:', X_test_scaled.shape)

## 7. Matriz de correlacion

In [ ]:
cols_corr = ['cancelacion', 'meses_contratado', 'cargo_mensual', 'cargo_total',
             'cargo_diario', 'total_servicios', 'adulto_mayor', 'pareja',
             'dependientes', 'factura_digital']

corr = df_encoded[cols_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, square=True, annot_kws={'size': 9})
ax.set_title('Matriz de correlacion — variables numericas clave', fontweight='bold')
plt.tight_layout()
plt.show()

print('Correlacion con cancelacion (desc):')
print(corr['cancelacion'].drop('cancelacion').sort_values(ascending=False).round(4).to_string())

## 8. Relacion de variables clave con la cancelacion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
activo  = df_encoded[df_encoded['cancelacion'] == 0]
cancelo = df_encoded[df_encoded['cancelacion'] == 1]
color_a, color_c = '#2ecc71', '#e74c3c'

# Boxplot: meses_contratado
data_meses = [activo['meses_contratado'], cancelo['meses_contratado']]
bp1 = axes[0].boxplot(data_meses, patch_artist=True, widths=0.5,
                      medianprops={'color': 'black', 'linewidth': 2})
bp1['boxes'][0].set_facecolor(color_a)
bp1['boxes'][1].set_facecolor(color_c)
axes[0].set_xticklabels(['Activo', 'Cancelo'])
axes[0].set_title('Tiempo de contrato vs Cancelacion', fontweight='bold')
axes[0].set_ylabel('Meses contratado')

# Boxplot: cargo_total
data_gasto = [activo['cargo_total'], cancelo['cargo_total']]
bp2 = axes[1].boxplot(data_gasto, patch_artist=True, widths=0.5,
                      medianprops={'color': 'black', 'linewidth': 2})
bp2['boxes'][0].set_facecolor(color_a)
bp2['boxes'][1].set_facecolor(color_c)
axes[1].set_xticklabels(['Activo', 'Cancelo'])
axes[1].set_title('Gasto total vs Cancelacion', fontweight='bold')
axes[1].set_ylabel('Cargo total')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(activo['meses_contratado'],  activo['cargo_total'],
           alpha=0.3, s=12, color=color_a, label='Activo',  edgecolors='none')
ax.scatter(cancelo['meses_contratado'], cancelo['cargo_total'],
           alpha=0.3, s=12, color=color_c, label='Cancelo', edgecolors='none')
ax.set_xlabel('Meses contratado')
ax.set_ylabel('Cargo total')
ax.set_title('Tiempo de contrato vs Gasto total por estado del cliente', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Entrenamiento de modelos

### Modelo 1 — Regresion Logistica (con normalizacion)

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train_bal)
y_pred_lr = lr.predict(X_test_scaled)
print('Regresion Logistica — entrenamiento completado')

### Modelo 2 — KNN (con normalizacion)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=11)
knn.fit(X_train_scaled, y_train_bal)
y_pred_knn = knn.predict(X_test_scaled)
print('KNN — entrenamiento completado')

### Modelo 3 — Random Forest (sin normalizacion)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_bal, y_train_bal)
y_pred_rf = rf.predict(X_test_raw)
print('Random Forest — entrenamiento completado')

### Modelo 4 — SVM (con normalizacion)

In [ ]:
svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train_scaled, y_train_bal)
y_pred_svm = svm.predict(X_test_scaled)
print('SVM — entrenamiento completado')

## 10. Evaluacion y comparacion de modelos

In [ ]:
def evaluar_modelo(nombre, y_true, y_pred):
    print(f'\n{"="*52}')
    print(f'  {nombre}')
    print(f'{"="*52}')
    print(f'  Exactitud : {accuracy_score(y_true, y_pred):.4f}')
    print(f'  Precision : {precision_score(y_true, y_pred):.4f}')
    print(f'  Recall    : {recall_score(y_true, y_pred):.4f}')
    print(f'  F1-score  : {f1_score(y_true, y_pred):.4f}')
    print()
    print(classification_report(y_true, y_pred, target_names=['Activo', 'Cancelo']))

evaluar_modelo('Regresion Logistica', y_test, y_pred_lr)
evaluar_modelo('KNN (k=11)',          y_test, y_pred_knn)
evaluar_modelo('Random Forest',       y_test, y_pred_rf)
evaluar_modelo('SVM (RBF)',           y_test, y_pred_svm)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, nombre, y_pred in zip(
    axes,
    ['Regresion Logistica', 'KNN (k=11)', 'Random Forest', 'SVM (RBF)'],
    [y_pred_lr, y_pred_knn, y_pred_rf, y_pred_svm]
):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Activo', 'Cancelo'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(nombre, fontweight='bold')

fig.suptitle('Matrices de confusion — todos los modelos', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
resultados = pd.DataFrame({
    'Modelo':    ['Regresion Logistica', 'KNN (k=11)', 'Random Forest', 'SVM (RBF)'],
    'Exactitud': [accuracy_score(y_test, p)  for p in [y_pred_lr, y_pred_knn, y_pred_rf, y_pred_svm]],
    'Precision': [precision_score(y_test, p) for p in [y_pred_lr, y_pred_knn, y_pred_rf, y_pred_svm]],
    'Recall':    [recall_score(y_test, p)    for p in [y_pred_lr, y_pred_knn, y_pred_rf, y_pred_svm]],
    'F1-score':  [f1_score(y_test, p)        for p in [y_pred_lr, y_pred_knn, y_pred_rf, y_pred_svm]],
}).set_index('Modelo').round(4)

print(resultados.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
resultados.T.plot(kind='bar', ax=ax, edgecolor='black', alpha=0.85)
ax.set_title('Comparacion de metricas entre modelos', fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.tick_params(axis='x', rotation=0)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 12. Analisis de variables relevantes por modelo

### Regresion Logistica — coeficientes

In [ ]:
coef_lr = pd.Series(lr.coef_[0], index=X.columns).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

coef_lr.head(15).plot(kind='barh', ax=axes[0], color='#3498db', edgecolor='black', alpha=0.85)
axes[0].set_title('Top 15 — mayor efecto protector (menos churn)', fontweight='bold')
axes[0].axvline(0, color='black', linewidth=0.8)

coef_lr.tail(15).sort_values(ascending=False).plot(
    kind='barh', ax=axes[1], color='#e74c3c', edgecolor='black', alpha=0.85)
axes[1].set_title('Top 15 — mayor efecto de riesgo (mas churn)', fontweight='bold')
axes[1].axvline(0, color='black', linewidth=0.8)

fig.suptitle('Coeficientes — Regresion Logistica', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Top 10 variables de mayor riesgo (coeficiente positivo):')
print(coef_lr.sort_values(ascending=False).head(10).round(4).to_string())

### KNN — analisis de proximidad por variable

In [ ]:
# En KNN no hay coeficientes directos. Se evalua la importancia de variables
# mediante permutation importance: se permuta cada variable y se mide la caida en F1.
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    knn, X_test_scaled, y_test,
    n_repeats=10, random_state=42, scoring='f1'
)

importancia_knn = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
importancia_knn.head(20).sort_values().plot(
    kind='barh', ax=ax, color='#9b59b6', edgecolor='black', alpha=0.85)
ax.set_title('Top 20 variables por importancia de permutacion — KNN', fontweight='bold')
ax.set_xlabel('Caida media en F1 al permutar la variable')
plt.tight_layout()
plt.show()

print('Top 10 variables mas influyentes en KNN:')
print(importancia_knn.head(10).round(4).to_string())

### Random Forest — importancia por reduccion de impureza

In [ ]:
importancias_rf = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
importancias_rf.head(20).sort_values().plot(
    kind='barh', ax=ax, color='#27ae60', edgecolor='black', alpha=0.85)
ax.set_title('Top 20 variables por importancia — Random Forest', fontweight='bold')
ax.set_xlabel('Importancia (reduccion media de impureza de Gini)')
plt.tight_layout()
plt.show()

print('Top 10 variables mas importantes en Random Forest:')
print(importancias_rf.head(10).round(4).to_string())

### SVM — coeficientes de soporte (kernel lineal para interpretabilidad)

In [ ]:
# El SVM con kernel RBF no provee coeficientes directos.
# Se entrena un SVM lineal adicional solo para analisis de variables.
svm_lineal = SVC(kernel='linear', random_state=42, max_iter=2000)
svm_lineal.fit(X_train_scaled, y_train_bal)

coef_svm = pd.Series(svm_lineal.coef_[0], index=X.columns).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

coef_svm.head(15).plot(kind='barh', ax=axes[0], color='#3498db', edgecolor='black', alpha=0.85)
axes[0].set_title('Top 15 — efecto protector', fontweight='bold')
axes[0].axvline(0, color='black', linewidth=0.8)

coef_svm.tail(15).sort_values(ascending=False).plot(
    kind='barh', ax=axes[1], color='#e74c3c', edgecolor='black', alpha=0.85)
axes[1].set_title('Top 15 — efecto de riesgo', fontweight='bold')
axes[1].axvline(0, color='black', linewidth=0.8)

fig.suptitle('Coeficientes — SVM lineal (para interpretabilidad)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Top 10 variables de mayor riesgo segun SVM lineal:')
print(coef_svm.sort_values(ascending=False).head(10).round(4).to_string())

### Comparacion de rankings de variables entre modelos

In [ ]:
# Normalizar importancias de cada modelo a escala 0-1 para comparar
def normalizar(serie):
    s = serie.abs()
    return (s / s.max()).round(4)

top_vars = list(importancias_rf.head(10).index)

ranking = pd.DataFrame({
    'Random Forest':         normalizar(importancias_rf)[top_vars],
    'Reg. Logistica':        normalizar(coef_lr.abs())[top_vars],
    'SVM lineal':            normalizar(coef_svm.abs())[top_vars],
    'KNN (permutacion)':     normalizar(importancia_knn)[top_vars],
}).sort_values('Random Forest', ascending=False)

fig, ax = plt.subplots(figsize=(11, 6))
ranking.plot(kind='bar', ax=ax, edgecolor='black', alpha=0.85)
ax.set_title('Importancia relativa de variables top-10 segun cada modelo', fontweight='bold')
ax.set_ylabel('Importancia normalizada (0-1)')
ax.tick_params(axis='x', rotation=35)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(ranking.to_string())

## 13. Informe final — Factores de cancelacion y estrategias de retencion

---

### Factores que mas influyen en la cancelacion

A partir del analisis de coeficientes y de importancia de variables en los cuatro modelos, los factores que aparecen de forma consistente como los mas relevantes para predecir la cancelacion son:

**1. Tipo de contrato mes a mes**
Es la variable con mayor peso en todos los modelos. Los clientes sin compromiso de permanencia tienen una probabilidad de cancelacion entre 3 y 14 veces mayor que quienes tienen contratos anuales o bianuales. La ausencia de friccion para cancelar es el principal driver del churn.

**2. Antiguedad del cliente (meses_contratado)**
Variable con correlacion negativa fuerte: a mayor tiempo en la empresa, menor probabilidad de cancelacion. Los primeros 12 meses son el periodo de mayor riesgo. Esto indica que el problema no es solo de precio sino de falta de vinculacion inicial.

**3. Servicio de internet por fibra optica**
Los clientes de fibra optica cancelan en mayor proporcion que los de DSL o sin internet, a pesar de pagar mas. Esto sugiere una brecha entre la expectativa generada por el producto y la experiencia real del servicio.

**4. Cargo mensual elevado**
A mayor cargo mensual, mayor probabilidad de cancelacion. El efecto se amplifica cuando el cliente percibe que no recibe valor proporcional al costo, especialmente en los primeros meses.

**5. Metodo de pago: cheque electronico**
Aparece como factor de riesgo en los modelos lineales. Puede ser un indicador de clientes menos comprometidos con la automatizacion de pagos, o de mayor friccion en la experiencia de facturacion.

**6. Ausencia de servicios adicionales**
Los clientes sin seguridad online, backup, soporte tecnico o streaming tienen mayor tasa de cancelacion. Cada servicio adicional actua como un factor de retencion porque aumenta el costo percibido de cambiar de proveedor.

**7. Factura digital**
Correlaciona positivamente con el churn, posiblemente porque los clientes digitales tienen mayor facilidad para comparar ofertas de la competencia y gestionar cancelaciones online.

---

### Comparacion de modelos

| Modelo | Fortaleza | Limitacion |
|---|---|---|
| Random Forest | Mejor F1 general, captura no linealidades, robusto | Menor interpretabilidad directa |
| Regresion Logistica | Interpretable, coeficientes directos | Asume linealidad, menor F1 en clases desbalanceadas |
| SVM (RBF) | Buen rendimiento en espacios de alta dimension | Alto costo computacional, dificil de interpretar |
| KNN | Simple, no paramétrico | Sensible al ruido, lento en produccion, sin importancia directa |

El modelo recomendado para produccion es **Random Forest** por su equilibrio entre rendimiento y estabilidad. Para comunicacion con stakeholders no tecnicos, la **Regresion Logistica** ofrece mayor claridad en la interpretacion de factores de riesgo.

---

### Estrategias de retencion recomendadas

**Contratos y fidelizacion**
Ofrecer descuentos progresivos o beneficios concretos (meses gratis, upgrades) para migrar clientes de contratos mes a mes a planes anuales. El incentivo debe ser mayor en los primeros 3 meses, que es cuando el riesgo de abandono es mas alto.

**Programa de onboarding**
Implementar un acompanamiento activo durante los primeros 90 dias: contacto proactivo a los 7, 30 y 60 dias, tutoriales de uso, soporte prioritario. El objetivo es construir habito de uso antes de que el cliente evalúe la cancelacion.

**Fibra optica: experiencia de servicio**
Auditar la calidad del servicio de fibra optica y contrastar los indicadores tecnicos con la satisfaccion declarada. Crear un plan de retencion especifico para este segmento con SLA garantizado o compensacion ante interrupciones.

**Estrategia de bundle**
Promover la contratacion de servicios adicionales (seguridad online, backup, streaming) mediante precios de paquete. Cada servicio adicional eleva el costo de cambio percibido y reduce la probabilidad de cancelacion.

**Automatizacion de pagos**
Incentivar la migracion desde cheque electronico hacia debito automatico o tarjeta de credito con un descuento puntual. Reducir la friccion en el proceso de pago disminuye también el riesgo de cancelacion involuntaria.

**Segmentacion predictiva**
Usar el modelo de Random Forest en produccion para generar un score de riesgo mensual por cliente. Los clientes con score alto deben recibir una intervencion comercial proactiva (llamada, oferta de retencion, encuesta de satisfaccion) antes de que inicien el proceso de cancelacion.

## 11. Analisis critico de los modelos

### Preprocesamiento aplicado

- **Eliminacion de columnas**: se removio `id_cliente` por ser un identificador unico sin capacidad predictiva.
- **One-Hot Encoding**: las variables categoricas (genero, tipo de contrato, metodo de pago, servicio de internet) fueron convertidas a columnas binarias. Se mantuvo `drop_first=False` para conservar legibilidad; en produccion se puede activar para evitar multicolinealidad perfecta.
- **Balanceo con SMOTE**: el dataset presenta un desbalance de aproximadamente 74/26 entre clientes activos y cancelados. Se aplico SMOTE sobre el conjunto de entrenamiento para generar ejemplos sinteticos de la clase minoritaria, evitando que el modelo se sesgue hacia predecir siempre "activo".
- **Normalizacion**: se aplico `StandardScaler` exclusivamente para la Regresion Logistica, ya que este modelo optimiza coeficientes mediante gradiente y es sensible a la escala. Random Forest no requiere normalizacion porque toma decisiones basadas en umbrales de corte, no en distancias ni magnitudes.
- **Division train/test**: 80% entrenamiento, 20% prueba con estratificacion para mantener la proporcion de clases en ambos conjuntos.

---

### Comparacion de modelos

**Regresion Logistica**

Modelo lineal de bajo costo computacional. Funciona bien cuando existe una relacion aproximadamente lineal entre las variables y el target. Es interpretable: los coeficientes indican la direccion e intensidad del efecto de cada variable. Su limitacion principal es que no captura interacciones no lineales entre variables.

**Random Forest**

Ensemble de arboles de decision que captura relaciones no lineales e interacciones entre variables sin necesidad de normalizacion. Tiende a generalizar mejor en datasets con muchas variables y relaciones complejas. Adicionalmente, provee un ranking de importancia de variables util para etapas posteriores de seleccion de features.

---

### Overfitting / Underfitting

Si Random Forest muestra una exactitud notablemente superior en entrenamiento que en prueba, puede indicar overfitting. Para mitigarlo se puede reducir `max_depth`, aumentar `min_samples_leaf` o reducir `n_estimators`. La Regresion Logistica es menos propensa al overfitting por su simplicidad, pero puede presentar underfitting si las relaciones en los datos son no lineales.

---

### Conclusion

El modelo con mejor desempeño esperado en este tipo de problema es Random Forest, principalmente por su capacidad de capturar interacciones no lineales y su robustez ante variables de diferente escala y tipo. Sin embargo, la Regresion Logistica ofrece mayor interpretabilidad, lo que puede ser valioso para comunicar resultados al area comercial. En un contexto de negocio, el Recall sobre la clase "Cancelo" es la metrica mas critica: es preferible identificar correctamente a los clientes en riesgo aunque se generen algunas falsas alarmas.